In [1]:
import torch
import mic21_model
from mic21_model.configuration_mic21 import MIC21SummarizerConfig
from mic21_model.modeling_mic21 import MIC21SummarizerModel
from datasets import load_dataset
from transformers import TrainingArguments
from transformers import Trainer
import numpy as np
import pdb
from mic21_model.DistributedTrainer import DistributedTrainer

In [2]:
cuda_id = 2
device_map = torch.load('llama_3_dm.pth')
device_map['model.embed_tokens'] = 1
device_map['model.norm'] = 2
device_map['model.rotary_emb'] = 2
device_map['lm_head'] = 1
device_map['model.layers.12'] = 2
device_map['model.layers.13'] = 2
device_map['model.layers.14'] = 2
device_map['model.layers.15'] = 2
device_map['model.layers.16'] = 2
device_map['model.layers.17'] = 2
device_map['model.layers.18'] = 2
device_map['model.layers.19'] = 2
device_map['model.layers.20'] = 2
device_map['model.layers.21'] = 2
device_map['model.layers.22'] = 2
device_map['model.layers.23'] = 2
device_map['model.layers.0'] = 1
device_map['model.layers.1'] = 1
device_map['model.layers.2'] = 2
device_map['model.layers.3'] = 2

In [3]:
config = MIC21SummarizerConfig(
    device_map=device_map,
    memory_map={0: "3GiB", 1: "3GiB",2: "5GiB",},
    in_device = device_map['model.embed_tokens'],
    out_device = device_map['lm_head'],
    im_model_cuda_id = cuda_id)
model = MIC21SummarizerModel(config)
model.init_components()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!


In [4]:
dataset = load_dataset("jkralev/mic21_chess")

README.md:   0%|          | 0.00/523 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [4]:
model.save_pretrained('mic21_model')

In [ ]:
dataset = load_dataset("mic21_dataset")

domain_name = "chess"
indices = [i for i, label in enumerate(dataset['train']['label']) if label == domain_name]
dataset_subset = dataset['train'].select(indices)

In [7]:
dataset_subset.push_to_hub("jkralev/mic21_chess")

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Map:   0%|          | 0/124 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/jkralev/mic21_chess/commit/16a443c4cd8d52871ade46773c6943469b78ba1c', commit_message='Upload dataset', commit_description='', oid='16a443c4cd8d52871ade46773c6943469b78ba1c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/jkralev/mic21_chess', endpoint='https://huggingface.co', repo_type='dataset', repo_id='jkralev/mic21_chess'), pr_revision=None, pr_num=None)

In [4]:
dataset_subset = dataset_subset.remove_columns(['label','description','size','annotations'])

Using the latest cached version of the dataset since mic21_dataset couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/jordan/.cache/huggingface/datasets/mic21_dataset/default/0.0.0/1e7f85809b0f9e4e (last modified on Mon Nov 24 14:16:00 2025).


In [5]:
def data_collator_1(features):
    return {'images':[f['image'] for f in features], 'titles':[f['title'] for f in features]}

In [6]:
training_args = TrainingArguments(
    output_dir="mic21_model/train_cricket",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    remove_unused_columns=False,
)

In [7]:
trainer = DistributedTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_subset,
    eval_dataset=dataset_subset,
    data_collator=data_collator_1
)

Detected kernel version 4.15.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [8]:
trainer.train()

`OffloadedCache` is deprecated and will be removed in version v4.59 Use `DynamicCache(offloading=True)` instead
/usr/local/lib/python3.10/dist-packages/torch/autograd/graph.py:744: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Epoch,Training Loss,Validation Loss


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0! (when checking argument for argument weight in method wrapper_CUDA__native_layer_norm)

In [4]:
model.save_pretrained('mic21_model/test_ch')

In [5]:
model.push_to_hub('jkralev/mic21_model')

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/jkralev/mic21_model/commit/0d12ceb2c8e2b550077a6f3bc333a717354af01e', commit_message='Upload model', commit_description='', oid='0d12ceb2c8e2b550077a6f3bc333a717354af01e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jkralev/mic21_model', endpoint='https://huggingface.co', repo_type='model', repo_id='jkralev/mic21_model'), pr_revision=None, pr_num=None)

In [8]:
dir(mic21_model)

['AutoConfig',
 'AutoModel',
 'MIC21SummarizerConfig',
 'MIC21SummarizerModel',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 'configuration_mic21',
 'modeling_mic21']

In [1]:
from transformers import AutoModel,AutoConfig

model = AutoModel.from_pretrained("jkralev/mic21_model", trust_remote_code=True)

A new version of the following files was downloaded from https://huggingface.co/jkralev/mic21_model:
- configuration_mic21.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_mic21.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jkralev/mic21_model:
- modeling_mic21.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/410k [00:00<?, ?B/s]

In [7]:
model

MIC21SummarizerModel(
  (projection_layer): Linear(in_features=49, out_features=2048, bias=True)
  (projection_dropout): Dropout(p=0.1, inplace=False)
)

In [6]:
from detectron2 import model_zoo,config
from detectron2.modeling import build_model

from detectron2.modeling.meta_arch.rcnn import GeneralizedRCNN

cfg = config.get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml"))
#cfg.MODEL.WEIGHTS = config.detectron2_weights_file
#model_cfg = GeneralizedRCNN.from_config(cfg)
#model = GeneralizedRCNN(
#    backbone=model_cfg['backbone'],
#    proposal_generator=model_cfg['proposal_generator'],
#    roi_heads=model_cfg['roi_heads'],
#    pixel_mean=model_cfg['pixel_mean'],
#    pixel_std=model_cfg['pixel_std'],
#    input_format=model_cfg['input_format'])